# lf_io — schema & I/O dùng chung cho mọi Labeling Function

Mỗi LF ghi ra một file riêng `labels/votes/lf<N>_<tên>.csv` theo schema dưới đây. Nạp bằng `%run`:

```python
UTILS = ROOT / 'src' / 'utils' / 'lf_io.ipynb'
get_ipython().run_line_magic('run', str(UTILS))
```

**Schema (long/tidy — mỗi dòng = một phiếu cho một cặp (ảnh, tác vụ)):**

| cột | ý nghĩa |
|-----|---------|
| `lf` | tên LF sinh phiếu (vd `lf1_maturity`) |
| `image_id` | mã ảnh |
| `task` | một trong `TASKS` |
| `vote` | `1` hữu dụng / `0` không (abstain = **bỏ dòng**, không ghi) |
| `confidence` | độ tin (tuỳ chọn, để trống nếu không có) |
| `reason` | lý do (tuỳ chọn) |
| `source` | nguồn ảnh |
| `path` | đường dẫn tương đối |

Cột riêng của từng LF (vd `axis`, `delta` của LF6) truyền qua `extra_fields` và ghi sau các cột chuẩn.
Mỗi LF ghi `labels/votes/lf<N>_<tên>.csv`. Bước fusion đọc mọi file này bằng `fuse_votes`.

In [ ]:
import csv
import os
from pathlib import Path

# Danh sach tac vu chuan. Moi LF phai bo phieu theo dung cac ma nay.
TASKS = [
    "1_maturity_evaluation",
    "2_foliar_disease",
    "3_trunk_disease",
    "4_crown_disease",
    "5_petiole",
]

# Cot bat buoc cua mot phieu, theo thu tu. Cot extra cua tung LF ghi sau cac cot nay.
LF_VOTE_FIELDS = ["lf", "image_id", "task", "vote", "confidence", "reason", "source", "path"]

In [ ]:
def is_abstain(vote):
    if vote is None:
        return True
    if isinstance(vote, float) and vote != vote:
        return True
    if vote == "":
        return True
    return False


def make_vote(lf, image_id, task, vote, confidence="", reason="", source="", path="", **extra):
    """Tao mot dong phieu theo schema chuan. Tra None neu abstain (dong se bi bo)."""
    if is_abstain(vote):
        return None
    if task not in TASKS:
        raise ValueError("task khong hop le: " + repr(task))
    v = int(vote)
    if v not in (0, 1):
        raise ValueError("vote phai la 0/1, nhan " + repr(vote))
    row = {}
    row["lf"] = lf
    row["image_id"] = image_id
    row["task"] = task
    row["vote"] = v
    row["confidence"] = confidence
    row["reason"] = reason
    row["source"] = source
    row["path"] = path
    for k in extra:
        row[k] = extra[k]
    return row

In [ ]:
def write_lf_votes(out_path, rows, extra_fields=None):
    """Ghi phieu cua MOT LF ra file CSV RIENG (long schema), ghi atomic (tmp -> replace).

    out_path     : Path, vd labels/votes/lf1_maturity.csv (moi LF mot file khac nhau).
    rows         : list cac dict tu make_vote(); dong None (abstain) bi loai.
    extra_fields : list ten cot rieng cua LF, ghi sau cac cot chuan.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(LF_VOTE_FIELDS)
    if extra_fields is not None:
        for c in extra_fields:
            if c not in fields:
                fields.append(c)
    clean = []
    for r in rows:
        if r is None:
            continue
        clean.append(r)
    tmp = out_path.with_suffix(out_path.suffix + ".tmp")
    handle = tmp.open("w", newline="")
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    for r in clean:
        line = {}
        for k in fields:
            line[k] = r.get(k, "")
        writer.writerow(line)
    handle.close()
    os.replace(tmp, out_path)
    print_summary(out_path, clean)
    return out_path


def print_summary(out_path, rows):
    print("ghi:", out_path, "|", len(rows), "phieu")
    for t in TASKS:
        sub = []
        for r in rows:
            if r["task"] == t:
                sub.append(r)
        if not sub:
            continue
        pos = 0
        neg = 0
        for r in sub:
            if int(r["vote"]) == 1:
                pos = pos + 1
            else:
                neg = neg + 1
        print("  " + t.ljust(24) + ": 1=" + str(pos) + "  0=" + str(neg))

In [ ]:
def read_lf_votes(path):
    """Doc mot file phieu LF -> DataFrame."""
    import pandas as pd
    return pd.read_csv(path)


def fuse_votes(votes_dir, pattern="lf*.csv"):
    """Gop moi file phieu LF trong votes_dir -> (long_df, wide_df).

    long_df : moi dong = mot phieu (lf, image_id, task, vote).
    wide_df : index (image_id, task), cot = lf, gia tri = vote (NaN = abstain / khong co phieu).
    File thieu cot bat buoc bi bo qua.
    """
    import pandas as pd
    votes_dir = Path(votes_dir)
    frames = []
    for f in sorted(votes_dir.glob(pattern)):
        d = pd.read_csv(f)
        need = set(["lf", "image_id", "task", "vote"])
        if not need.issubset(set(d.columns)):
            continue
        frames.append(d[["lf", "image_id", "task", "vote"]])
    if not frames:
        raise SystemExit("Khong thay file phieu nao khop " + pattern + " trong " + str(votes_dir))
    long_df = pd.concat(frames, ignore_index=True)
    wide_df = long_df.pivot_table(index=["image_id", "task"], columns="lf", values="vote", aggfunc="first")
    return long_df, wide_df

## Fold cross-fitting dùng chung (LF1, LF6)

In [ ]:
import hashlib
import re


def original_id(image_id, source):
    # Gop x3 augment Roboflow ve anh goc: '010_jpg.rf.<hash>' -> '010'. Anh benh (Mendeley) -> chinh no.
    if source.startswith("coconut-veirf-v5"):
        return re.split(pattern=r"_jpg", string=image_id, maxsplit=1)[0]
    return image_id


def fold_of(orig_id, k, seed):
    # Gan fold tat dinh theo anh goc (md5 -> khong phu thuoc PYTHONHASHSEED). Chung cho moi LF.
    digest = hashlib.md5(f"{seed}:{orig_id}".encode()).hexdigest()
    return int(digest, 16) % k


## Ghi phiếu incremental + resume (long loop chống timeout)

In [ ]:
def append_lf_vote(out_path, row, extra_fields=None):
    # Ghi NGAY mot phieu (append + flush) de resume khi timeout. row=None (abstain) -> bo qua.
    if row is None:
        return
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(LF_VOTE_FIELDS)
    if extra_fields is not None:
        for c in extra_fields:
            if c not in fields:
                fields.append(c)
    is_new = not out_path.exists()
    handle = out_path.open("a", newline="")
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    if is_new:
        writer.writeheader()
    line = {}
    for k in fields:
        line[k] = row.get(k, "")
    writer.writerow(line)
    handle.flush()
    handle.close()


def done_image_ids(out_path):
    # Tap image_id da ghi (de resume). File chua co -> rong.
    out_path = Path(out_path)
    if not out_path.exists():
        return set()
    ids = set()
    handle = out_path.open()
    reader = csv.DictReader(handle)
    for r in reader:
        ids.add(r["image_id"])
    handle.close()
    return ids
